# Pump-In Preserves `G^2 = I` On A Vacuum Mode

Let `P = |chi><chi|` be one rank-1 Wannier projector and `Q = I-P`. In the covariance sign convention used here, an occupied mode has covariance `+P`, and an unoccupied/vacuum mode has covariance `-P`.

The deterministic pump-in/reset step for this mode is

`G' = Q G Q + P`.

If the input state is already vacuum in that mode and has no coherence between the mode and its complement,

`G P = P G = -P`,

then `G = -P + Q G Q`. If also `G^2 = I`, then `(QGQ)^2 = Q`, so

`(G')^2 = (QGQ + P)^2 = (QGQ)^2 + P = Q + P = I`.

So pump-in maps a pure Gaussian covariance to another pure Gaussian covariance when applied to a measured/reset vacuum mode.

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

def hermitize(A):
    return 0.5 * (A + A.conj().T)


def fro(A):
    return float(np.linalg.norm(A, ord="fro"))


def random_complex_matrix(n, m, rng):
    return rng.normal(size=(n, m)) + 1j * rng.normal(size=(n, m))


def rank1_projector(v):
    v = np.asarray(v, dtype=np.complex128).reshape(-1)
    v = v / np.linalg.norm(v)
    return np.outer(v, v.conj())


def random_rank1_projector(n, rng):
    return rank1_projector(random_complex_matrix(n, 1, rng).reshape(-1))


def complement_basis(P):
    evals, evecs = np.linalg.eigh(hermitize(P))
    return evecs[:, evals < 0.5]


def random_pure_vacuum_in_P(P, rng):
    """Pure covariance with the P mode unoccupied: G P = P G = -P and G^2 = I."""
    B = complement_basis(P)
    signs = rng.choice([-1.0, 1.0], size=B.shape[1])
    Gq = B @ np.diag(signs) @ B.conj().T
    return hermitize(-P + Gq)


def pump_in(G, P):
    Q = np.eye(G.shape[0], dtype=np.complex128) - P
    return hermitize(Q @ G @ Q + P)


def measure_rank1_branch(G, P, s):
    """Exact rank-1 measurement branch. s=+1 occupied, s=-1 unoccupied."""
    Q = np.eye(G.shape[0], dtype=np.complex128) - P
    denom = 1.0 + s * np.trace(G @ P)
    if abs(denom) < 1e-12:
        raise FloatingPointError(f"Measurement branch has near-zero Born weight: denom={denom}")
    out = s * P + Q @ G @ Q - s * (Q @ G @ P @ G @ Q) / denom
    return hermitize(out)


def random_pure_covariance(n, rng):
    Z = random_complex_matrix(n, n, rng)
    U, _ = np.linalg.qr(Z)
    signs = rng.choice([-1.0, 1.0], size=n)
    return hermitize(U @ np.diag(signs) @ U.conj().T)


def check_case(n=24, seed=0):
    rng = np.random.default_rng(seed)
    P = random_rank1_projector(n, rng)
    I = np.eye(n, dtype=np.complex128)
    G = random_pure_vacuum_in_P(P, rng)
    Gp = pump_in(G, P)
    return {
        "input_purity_error": fro(G @ G - I),
        "vacuum_condition_error": fro(G @ P + P) + fro(P @ G + P),
        "output_purity_error": fro(Gp @ Gp - I),
        "output_occupied_error": fro(Gp @ P - P) + fro(P @ Gp - P),
        "hermiticity_error": fro(Gp - Gp.conj().T),
    }


rows = [check_case(n=32, seed=s) for s in range(5)]
for i, row in enumerate(rows):
    print(f"case {i}")
    for key, val in row.items():
        print(f"  {key:24s} {val:.3e}")
    assert row["input_purity_error"] < 1e-12
    assert row["vacuum_condition_error"] < 1e-12
    assert row["output_purity_error"] < 1e-12
    assert row["output_occupied_error"] < 1e-12

## Measure Unoccupied First, Then Pump In

Now start from an arbitrary pure covariance `G` that generally does **not** commute with `P`. First condition on the unoccupied measurement outcome, producing `G_meas`. This branch removes the `P-Q` coherences and sets the measured mode to `-P`. Then pump in with `G_out = Q G_meas Q + P`.

This is the physical sequence needed for the purity argument.

In [ ]:
def measured_then_pump_case(n=24, seed=0):
    rng = np.random.default_rng(seed)
    P = random_rank1_projector(n, rng)
    I = np.eye(n, dtype=np.complex128)
    G = random_pure_covariance(n, rng)

    G_meas = measure_rank1_branch(G, P, s=-1)  # find the mode unoccupied
    G_out = pump_in(G_meas, P)                 # then fill/pump in
    G_bare = pump_in(G, P)                     # wrong order: pump without measuring

    return {
        "input_purity_error": fro(G @ G - I),
        "input_vacuum_condition_error": fro(G @ P + P) + fro(P @ G + P),
        "measured_purity_error": fro(G_meas @ G_meas - I),
        "measured_unoccupied_error": fro(G_meas @ P + P) + fro(P @ G_meas + P),
        "pump_after_measure_purity_error": fro(G_out @ G_out - I),
        "pump_after_measure_occupied_error": fro(G_out @ P - P) + fro(P @ G_out - P),
        "bare_pump_purity_error": fro(G_bare @ G_bare - I),
    }


rows = [measured_then_pump_case(n=32, seed=s) for s in range(5)]
for i, row in enumerate(rows):
    print(f"case {i}")
    for key, val in row.items():
        print(f"  {key:36s} {val:.3e}")
    assert row["input_purity_error"] < 1e-12
    assert row["input_vacuum_condition_error"] > 1e-3
    assert row["measured_purity_error"] < 1e-12
    assert row["measured_unoccupied_error"] < 1e-12
    assert row["pump_after_measure_purity_error"] < 1e-12
    assert row["pump_after_measure_occupied_error"] < 1e-12
    assert row["bare_pump_purity_error"] > 1e-3

## Negative Control

The condition that the mode is a vacuum/decoupled mode matters. If `G^2 = I` but `G` has coherence with `P`, then the bare reset `QGQ + P` need not preserve `G^2 = I`. In the full measurement-plus-pump-in procedure, the measurement branch first removes those coherences and puts the mode into the definite unoccupied state; then pump-in flips it to occupied.

In [ ]:
def random_pure_covariance(n, rng):
    Z = random_complex_matrix(n, n, rng)
    U, _ = np.linalg.qr(Z)
    signs = rng.choice([-1.0, 1.0], size=n)
    return hermitize(U @ np.diag(signs) @ U.conj().T)


rng = np.random.default_rng(123)
n = 24
P = random_rank1_projector(n, rng)
I = np.eye(n, dtype=np.complex128)
G = random_pure_covariance(n, rng)
Gp = pump_in(G, P)

print(f"arbitrary pure input: ||G^2-I||_F       = {fro(G @ G - I):.3e}")
print(f"vacuum condition error                 = {fro(G @ P + P) + fro(P @ G + P):.3e}")
print(f"after bare pump-in: ||G'^2-I||_F        = {fro(Gp @ Gp - I):.3e}")
assert fro(G @ G - I) < 1e-12
assert fro(Gp @ Gp - I) > 1e-3